In [ ]:
import geopandas as gpd

In [2]:
# def tessellate(gdf: geopandas.GeoDataFrame, columns=["shapeName"], resolution=7):
#     """
#     Tessellates the geometries into H3 indexes in a GeoDataFrame

#     Parameters
#     ----------
#     gdf : geopandas.GeoDataFrame
#         The GeoDataFrame ()containing the geometries to tessellate.
#     column : str, optional
#         The column name in `gdf` to use in the resulting GeoDataFrame.
#         Default is "shapeName".
#     resolution : int, optional
#         The H3 resolution level for tessellation. Higher resolution results in smaller hexagons.
#         Default is 7.

#     Returns
#     -------
#     geopandas.GeoDataFrame
#         A GeoDataFrame containing the tessellated hexagons with the specified `index` and geometry.

#     Raises
#     ------
#     Exception
#         If a geometry type other than "Polygon" or "MultiPolygon" is encountered.

#     Examples
#     --------
#     >>> import geopandas as gpd
#     >>> from shapely.geometry import Polygon
#     >>> gdf = gpd.GeoDataFrame({
#     ...     'geometry': [Polygon([(0, 0), (1, 0), (1, 1), (0, 1)])],
#     ...     'shapeName': ['A']
#     ... })
#     >>> tessellate(gdf)
#           shapeName                                           geometry
#     8a69a0f7fffffff      POLYGON ((0 0, 0.5 0, 1 0.5, 1 1, 0.5 1, 0 1, 0 0))
#     """




In [8]:
import geopandas as gpd
import pandas as pd
from shapely.geometry import Polygon, MultiPolygon
import h3

In [6]:
gdf = gpd.read_file('../data/geography-levels/zonas-montevideo-actualizadas-steer.gpkg')
columns=["poligono_id"]
        #  , "tipo_zona", 'depto_codigo', 'depto_nombre', 'seccion_codigo', 'CODSEG']
resolution=7

In [24]:
gdf = gdf.to_crs("EPSG:4326")

# Use overlap containment so hexes covering small/thin zones aren't dropped
# (default center-based mode misses any polygon smaller than an h3 cell).
hex_ids = set()
uncovered = []
for idx, geometry in gdf.geometry.items():
    shape = h3.geo_to_h3shape(geometry)
    cells = h3.h3shape_to_cells_experimental(shape, resolution, contain="overlap")
    if not cells:
        uncovered.append(idx)
    hex_ids.update(cells)

print(f"zones with no covering hex: {len(uncovered)} / {len(gdf)}")

# `cell_to_boundary` returns (lat, lng) tuples; shapely expects (lng, lat).
gdf_new = gpd.GeoDataFrame(
    {"hex_id": list(hex_ids)},
    geometry=[

        Polygon([(lng, lat) for lat, lng in h3.cell_to_boundary(h)])# return gdf

        for h in hex_ids

    ],)

zones with no covering hex: 0 / 17503


In [25]:
gdf_new.to_file("h3_layer_test.gpkg")

/Users/mtadeo/Library/CloudStorage/OneDrive-WBG/Documents/gitrepo/montevideo-brt/.venv/lib/python3.14/site-packages/pyogrio/geopandas.py:917: UserWarning: 'crs' was not provided.  The output dataset will not have projection information defined and may not be usable in other systems.
  write(


In [17]:
gdf.reset_index(inplace = True)

In [19]:
gdf[gdf.index.duplicated()]

,index,poligono_id,geometry
